# tom — full training pipeline on Colab (T4 GPU) · **Drive-free version**

**This notebook NEVER touches Google Drive.** No mount, no sync — everything lives on the
Colab VM's local disk (~100 GB free) and the small result files are downloaded to your
computer at the end. Total ~2.5–3 GPU-hours.

**Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**, keep the tab open.

Trade-off vs the old version: if Colab disconnects mid-run, progress is lost and you re-run
from scratch (~2.5 h). In exchange, your Drive quota — and anything shared from it — cannot
be affected. The old Drive-sync design filled a 15 GB Drive with optimizer checkpoints;
this version cannot.

In [ ]:
# 1 · GPU check
import torch
!nvidia-smi -L
assert torch.cuda.is_available(), "No GPU! Runtime -> Change runtime type -> T4 GPU"
print("CUDA OK:", torch.cuda.get_device_name(0))

In [ ]:
# 2 · Clone the repo (local disk only — no Drive anywhere in this notebook)
%cd /content
!test -d COMP9444_26T2_FOMC_Analysis || git clone --depth 1 https://github.com/StrawHatSWE/COMP9444_26T2_FOMC_Analysis
%cd COMP9444_26T2_FOMC_Analysis/tom

In [ ]:
# 3 · Dependencies (torch preinstalled)
!pip install -q transformers datasets openpyxl "accelerate>=0.26" 

## Stage 1 — DAPT (~40–60 min)

In [ ]:
!python dapt.py --corpus fed_corpus.txt

## Stage 2 — Hyperparameter search (~30–45 min)

In [ ]:
!python search.py --model runs/dapt
!cat runs/search/selected_config.json

## Stage 3 — Final training, 3 seeds × 3 restarts (~30–60 min)

In [ ]:
!python train_final.py --restarts 3

## Stage 4 — Ensemble (~2 min)

In [ ]:
!python ensemble.py --device cuda

In [ ]:
# 5 · Results summary
import json
from pathlib import Path
single = json.loads(Path('runs/final/results.json').read_text())
ens = json.loads(Path('runs/final/results_ensemble.json').read_text())
print(f"single-model  F1: {single['single_model_mean_f1']:.4f} +/- {single['single_model_std_f1']:.4f}")
print(f"ensemble      F1: {ens['mean_f1']:.4f} +/- {ens['std_f1']:.4f}   acc {ens['mean_accuracy']:.4f}")
print("
baselines: tuned FinBERT 0.629 +/- 0.011 | published ceiling (RoBERTa-large) 0.71-0.74")
print("
per seed:", json.dumps(ens['per_seed'], indent=2))

In [ ]:
# 6 · Bank per-model test predictions (for confusion matrices / per-class analysis)
import pandas as pd, json, glob
import common
preds = {}
for d in sorted(glob.glob('runs/final/model-*')):
    seed = d.split('-')[1]
    test = pd.read_excel(common.data_dirs(False)[1] / f'lab-manual-combine-test-{seed}.xlsx')
    probs = common.predict_proba(d, test['sentence'].astype(str).tolist(), device='cuda')
    preds[d.split('/')[-1]] = {'probs': probs.tolist(), 'true': test['label'].tolist()}
json.dump(preds, open('/content/tom_predictions.json', 'w'))
print('banked predictions for', len(preds), 'models')

In [ ]:
# 7 · Package ALL small artifacts (~1 MB) and download to YOUR computer (not Drive)
!mkdir -p /content/tom_results
!cp runs/search/validation_search.csv runs/search/selected_config.json /content/tom_results/
!cp runs/final/results.json runs/final/results_ensemble.json /content/tom_results/
!cp runs/final/metrics-*.json /content/tom_results/ 2>/dev/null || true
!cp /content/tom_predictions.json /content/tom_results/
!cd /content && zip -q -r tom_results.zip tom_results
from google.colab import files
files.download('/content/tom_results.zip')
print('downloaded tom_results.zip — this file is everything the analysis needs')

## Optional — keep the trained models themselves
The zip above contains every number and prediction; models are only needed for live demos or
future reuse. If you want them, run the cell below to download the best seed's trio (~1.5 GB
total, three separate downloads). **Nothing here touches Drive either.**

```
!cd runs/final && zip -q -r /content/models_5768.zip model-5768-r0 model-5768-r1 model-5768-r2
from google.colab import files; files.download('/content/models_5768.zip')
```